# Práctica 13 — Test A/B y diseño de experimentos

**Curso:** Estadística Aplicada con Python · **Bloque 4:** Modelos y experimentos · **Tiempo estimado:** 60 minutos
**Clase asociada:** `notebooks_clase/clase13_ab_testing_diseno_experimentos.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una celda con `# TODO`: complétala.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te dice qué revisar. Si olvidas completar un `TODO`, el mensaje te lo recuerda.
- Los ejercicios con simulación usan tolerancias basadas en el error estándar, así que un resultado correcto no debería fallar por casualidad.
- Las **soluciones** están al final del notebook, en una sección aparte. **No las abras antes de haberlo intentado.**
- Ejecuta las celdas en orden. Todos los datos son sintéticos: *ejemplos ilustrativos*.

---
## Celda 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize, confint_proportions_2indep
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

print("Librerías y funciones cargadas correctamente.")

## Ejercicio 1 — Tamaño de muestra, a mano y con statsmodels

Una tienda con conversión del 8 % quiere detectar una mejora a 10 % (MDE de +2 puntos) con α = 0,05 y potencia del 90 %. Calcula: `za` y `zb` (cuantiles normales), `n_mano` (fórmula a mano, sin redondear), `h` (tamaño de efecto de `proportion_effectsize`, con el valor objetivo primero), `n_sm` (con `NormalIndPower().solve_power`), `n_grupo` (`n_sm` redondeado hacia arriba, entero), `dias` (días necesarios con 1 000 visitantes al día en total, repartidos entre los dos grupos, redondeado hacia arriba) y `semanas` (`dias` en semanas completas).

In [ ]:
p_base, p_obj = 0.08, 0.10
alfa, potencia = 0.05, 0.90
visitantes_dia = 1000

za = None        # TODO: cuantil z(1 − α/2)
zb = None        # TODO: cuantil z(potencia)
n_mano = None    # TODO: (za + zb)² · [p1(1−p1) + p2(1−p2)] / (p2 − p1)²
h = None         # TODO
n_sm = None      # TODO
n_grupo = None   # TODO
dias = None      # TODO
semanas = None   # TODO

# Comprobación automática
assert all(v is not None for v in (za, zb, n_mano, h, n_sm, n_grupo, dias, semanas)), "Completa las líneas marcadas con TODO"
assert np.isclose(za, 1.959964, atol=1e-4) and np.isclose(zb, 1.281552, atol=1e-4), "za = norm.ppf(1 - alfa/2); zb = norm.ppf(potencia)"
ref_mano = (za + zb) ** 2 * (0.08 * 0.92 + 0.10 * 0.90) / 0.02 ** 2
assert np.isclose(n_mano, ref_mano), "revisa la fórmula: (za + zb)² · suma de varianzas / diferencia²"
assert np.isclose(h, proportion_effectsize(0.10, 0.08)), "proportion_effectsize(p_obj, p_base)"
assert abs(n_sm - n_mano) < 0.01 * n_mano, "n_sm y n_mano deberían diferir menos de un 1 %"
assert n_grupo == int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.10, 0.08), alpha=0.05, power=0.90, ratio=1))), "n_grupo = ceil(n_sm)"
assert dias == int(np.ceil(2 * n_grupo / 1000)) and semanas == int(np.ceil(dias / 7)), "días = ceil(2·n_grupo / visitantes_dia); semanas = ceil(días / 7)"
print(f"n a mano: {n_mano:.1f}   n statsmodels: {n_sm:.1f}   por grupo: {n_grupo}   días: {dias}   semanas: {semanas}")
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Analizar un test A/B: p-value, IC y lift

Resultado de un experimento: A convirtió 300 de 5 000 visitantes y B convirtió 370 de 5 000. Calcula: `dif` (tasa de B menos tasa de A, en proporción), `p_valor` (test de dos proporciones con `proportions_ztest`, poniendo B primero), `ic_inf` e `ic_sup` (IC 95 % **a mano**, con el error estándar sin combinar `√(pA(1−pA)/n + pB(1−pB)/n)` y 1,96), `lift` (mejora relativa de B respecto a A, en proporción), `significativo` (`True` si p < 0,05) y `incluye_bajo_mde` (`True` si el IC contiene valores por debajo del efecto mínimo relevante de 0,01, es decir, si `ic_inf < 0.01`).

In [ ]:
n_a = n_b = 5000
conv_a, conv_b = 300, 370

dif = None               # TODO
p_valor = None           # TODO
ic_inf = None            # TODO
ic_sup = None            # TODO
lift = None              # TODO
significativo = None     # TODO
incluye_bajo_mde = None  # TODO

# Comprobación automática
assert all(v is not None for v in (dif, p_valor, ic_inf, ic_sup, lift, significativo, incluye_bajo_mde)), "Completa las líneas marcadas con TODO"
assert np.isclose(dif, 0.014), "dif = 370/5000 − 300/5000"
assert np.isclose(p_valor, proportions_ztest([370, 300], [5000, 5000])[1]), "proportions_ztest([conv_b, conv_a], [n_b, n_a]) y toma el segundo valor"
ee_ref = np.sqrt(0.06 * 0.94 / 5000 + 0.074 * 0.926 / 5000)
assert np.isclose(ic_inf, 0.014 - 1.96 * ee_ref) and np.isclose(ic_sup, 0.014 + 1.96 * ee_ref), "IC = dif ± 1,96 · error estándar sin combinar"
assert np.isclose(lift, 370 / 300 - 1), "lift = pB / pA − 1"
assert significativo == True and incluye_bajo_mde == True, "es significativo (p ≈ 0,005) pero el IC llega por debajo de 0,01"
print(f"Diferencia: {dif * 100:.2f} pp  p-value = {p_valor:.4f}  IC 95 % = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp  lift = {lift * 100:.1f} %")
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Simular el peeking

Simulamos 2 000 experimentos A/A (misma tasa real, 10 %) con 5 controles de 768 visitantes nuevos por grupo cada uno. Ya está calculada la matriz `z` (2 000 filas, una por experimento; 5 columnas, una por control). Completa: `sig` (matriz booleana: |z| > 1,96), `tasa_peeking` (proporción de experimentos que son significativos en **algún** control) y `tasa_final` (proporción que es significativa **solo** mirando el último control).

In [ ]:
rng_e3 = np.random.default_rng(21)
nsim, controles, por = 2000, 5, 768
xa = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
xb = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
ns = por * np.arange(1, controles + 1)
pp = (xa + xb) / (2 * ns)
z = ((xb - xa) / ns) / np.sqrt(pp * (1 - pp) * 2 / ns)

sig = None            # TODO: matriz booleana |z| > 1,96
tasa_peeking = None   # TODO
tasa_final = None     # TODO

# Comprobación automática
assert all(v is not None for v in (sig, tasa_peeking, tasa_final)), "Completa las líneas marcadas con TODO"
assert sig.shape == (2000, 5) and sig.dtype == bool, "sig = np.abs(z) > 1.96 (matriz booleana 2000 × 5)"
assert np.isclose(tasa_peeking, (np.abs(z) > 1.96).any(axis=1).mean()), "usa sig.any(axis=1).mean(): significativo en algún control"
assert np.isclose(tasa_final, (np.abs(z[:, -1]) > 1.96).mean()), "usa sig[:, -1].mean(): solo el último control"
# Esperado: ≈ 5 % mirando una vez (± 3 errores estándar ≈ 1,5 pp) y ≈ 13 % con 5 controles
assert 0.035 < tasa_final < 0.065, "mirando solo al final la tasa debería rondar el 5 %"
assert 0.10 < tasa_peeking < 0.17, "con 5 controles la tasa real de falsos positivos debería rondar el 13 %"
print(f"Falsos positivos mirando una vez: {tasa_final * 100:.1f} %   mirando 5 veces: {tasa_peeking * 100:.1f} %")
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Control SRM con chi-cuadrado

Dos experimentos diseñados con reparto 50/50 registran estos tamaños: experimento 1, A = 25 300 y B = 24 200; experimento 2, A = 24 900 y B = 24 600. Para cada uno calcula el chi-cuadrado de bondad de ajuste con `stats.chisquare` (tamaños esperados iguales): `chi2_1`, `p_1`, `chi2_2`, `p_2`. Después `srm_1` y `srm_2` valen `True` si el p-value es menor que 0,001. Por último, `chi2_1_mano` calcula el chi-cuadrado del experimento 1 **a mano**: la suma de (observado − esperado)² / esperado.

In [ ]:
obs_1 = np.array([25300, 24200])
obs_2 = np.array([24900, 24600])

chi2_1 = None       # TODO
p_1 = None          # TODO
chi2_2 = None       # TODO
p_2 = None          # TODO
srm_1 = None        # TODO
srm_2 = None        # TODO
chi2_1_mano = None  # TODO

# Comprobación automática
assert all(v is not None for v in (chi2_1, p_1, chi2_2, p_2, srm_1, srm_2, chi2_1_mano)), "Completa las líneas marcadas con TODO"
assert np.isclose(chi2_1, 24.4444, atol=1e-3) and np.isclose(chi2_2, 1.8182, atol=1e-3), "stats.chisquare(observados) devuelve (chi2, p); el esperado por defecto es el reparto igual"
assert srm_1 == True and srm_2 == False, "el experimento 1 tiene SRM (p ≈ 0,0000008) y el 2 no (p ≈ 0,18)"
assert np.isclose(chi2_1_mano, chi2_1), "esperado = total / 2 = 24 750; suma de (obs − esp)² / esp para los dos grupos"
print(f"Experimento 1: chi2 = {chi2_1:.2f}  p = {p_1:.7f}  SRM = {srm_1}   |   Experimento 2: chi2 = {chi2_2:.2f}  p = {p_2:.3f}  SRM = {srm_2}")
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — La paradoja de Simpson

Un experimento con dos tipos de usuarios (`nuevos` y `recurrentes`). Con la tabla `tabla`, calcula: `tasa_seg` (DataFrame con la tasa de conversión de cada segmento —filas— y grupo —columnas—), `tasa_total` (Serie con la tasa total de cada grupo), `simpson` (`True` si B gana en **todos** los segmentos y pierde en el total) y `dif_estand` (diferencia B − A, en puntos porcentuales, usando como peso de cada segmento su proporción del total de usuarios en A y B juntos).

In [ ]:
tabla = pd.DataFrame({"segmento": ["nuevos", "nuevos", "recurrentes", "recurrentes"],
                      "grupo": ["A", "B", "A", "B"],
                      "usuarios": [200, 700, 800, 300],
                      "conversiones": [20, 84, 240, 96]})
conv = tabla.pivot(index="segmento", columns="grupo", values="conversiones")
usr = tabla.pivot(index="segmento", columns="grupo", values="usuarios")

tasa_seg = None     # TODO
tasa_total = None   # TODO
simpson = None      # TODO
dif_estand = None   # TODO

# Comprobación automática
assert all(v is not None for v in (tasa_seg, tasa_total, simpson, dif_estand)), "Completa las líneas marcadas con TODO"
assert np.isclose(tasa_seg.loc["nuevos", "A"], 0.10) and np.isclose(tasa_seg.loc["recurrentes", "B"], 0.32), "tasa_seg = conv / usr (segmentos en filas, grupos en columnas)"
assert np.isclose(tasa_total["A"], 0.26) and np.isclose(tasa_total["B"], 0.18), "tasa total = conv.sum() / usr.sum()"
assert simpson == True, "B gana en los dos segmentos (12 % > 10 %, 32 % > 30 %) y pierde en el total (18 % < 26 %)"
assert np.isclose(dif_estand, 2.0), "pesos = usuarios del segmento (A + B) / total; tasa estandarizada = suma(peso · tasa del segmento)"
print(tasa_seg.round(3).to_string())
print("Total:", (tasa_total * 100).round(1).to_dict(), "  diferencia estandarizada (pp):", round(dif_estand, 2))
print("Ejercicio 5 correcto ✔")

---
## Reto final — Informe de un experimento A/B

Una tienda planifica un test A/B: conversión actual del 8 %, mejora mínima a detectar de +1,5 puntos (hasta 9,5 %), α = 0,05, potencia del 80 %. El experimento ya se ha ejecutado y los datos están en `datos_reto` (columnas `grupo` y `convierte`). Elabora el informe: `n_plan` (tamaño por grupo planificado, con `NormalIndPower` y `proportion_effectsize`, redondeado hacia arriba), `tamanos` (diccionario con los usuarios reales de `A` y `B`), `p_srm` (p-value del chi-cuadrado de SRM sobre esos tamaños, reparto esperado 50/50), `p_valor` (test de dos proporciones, B primero), `ic_inf` e `ic_sup` (IC 95 % de la diferencia B − A, con `confint_proportions_2indep` método `"newcomb"`) y `decision`: `"SRM"` si hay SRM (p_srm < 0,001); en caso contrario `"GO"` (lanzar B) si p_valor < 0,05 y `ic_inf` > 0, y `"NO_GO"` en los demás casos. Escribe en `informe` un párrafo breve (mínimo 60 caracteres) con diseño, controles, resultado y decisión.

In [ ]:
def generar_experimento(rng, n_a=5561, n_b=5548, p_a=0.08, p_b=0.095):
    """Datos sintéticos (ejemplo ilustrativo): una fila por visitante."""
    convierte = np.concatenate([rng.random(n_a) < p_a, rng.random(n_b) < p_b]).astype(int)
    return pd.DataFrame({"grupo": ["A"] * n_a + ["B"] * n_b, "convierte": convierte})

datos_reto = generar_experimento(np.random.default_rng(95))

n_plan = None     # TODO
tamanos = None    # TODO
p_srm = None      # TODO
p_valor = None    # TODO
ic_inf = None     # TODO
ic_sup = None     # TODO
decision = None   # TODO
informe = None    # TODO

# Comprobación automática
assert all(v is not None for v in (n_plan, tamanos, p_srm, p_valor, ic_inf, ic_sup, decision, informe)), "Completa las líneas marcadas con TODO"
ref_n = int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1)))
assert n_plan == ref_n, "n_plan = ceil(NormalIndPower().solve_power(proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1))"
g = datos_reto.groupby("grupo")["convierte"].agg(["sum", "count"])
assert tamanos == {"A": int(g.loc["A", "count"]), "B": int(g.loc["B", "count"])}, "tamanos = {'A': usuarios de A, 'B': usuarios de B}"
assert np.isclose(p_srm, stats.chisquare([g.loc["A", "count"], g.loc["B", "count"]])[1]), "stats.chisquare sobre los dos tamaños, reparto esperado igual"
ref_p = proportions_ztest([g.loc["B", "sum"], g.loc["A", "sum"]], [g.loc["B", "count"], g.loc["A", "count"]])[1]
assert np.isclose(p_valor, ref_p), "proportions_ztest([conv_B, conv_A], [n_B, n_A]) y toma el segundo valor"
ref_lo, ref_hi = confint_proportions_2indep(g.loc["B", "sum"], g.loc["B", "count"], g.loc["A", "sum"], g.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf, ref_lo) and np.isclose(ic_sup, ref_hi), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb')"
assert p_srm > 0.001 and p_valor < 0.05 and ic_inf > 0, "en estos datos: sin SRM, significativo, y el IC no incluye el 0"
assert decision == "GO", "aplica la regla: SRM → revisar; p < 0,05 e ic_inf > 0 → lanzar B; si no, no lanzar B"
assert isinstance(informe, str) and len(informe) >= 60, "escribe un párrafo de al menos 60 caracteres"
print(f"n_plan = {n_plan}   tamaños: {tamanos}   p_srm = {p_srm:.3f}   p-value = {p_valor:.4f}   IC = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp")
print("Decisión:", decision)
print("Reto final correcto ✔")

---
## Cierre de la práctica

Has recorrido el ciclo de un test A/B: calcular el tamaño de muestra, analizar con IC y lift, medir el efecto del *peeking*, comprobar el SRM y reconocer la paradoja de Simpson. Continúa con la clase 14: Proyecto integrador.

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
p_base, p_obj = 0.08, 0.10
alfa, potencia = 0.05, 0.90
visitantes_dia = 1000

za = stats.norm.ppf(1 - alfa / 2)
zb = stats.norm.ppf(potencia)
n_mano = (za + zb) ** 2 * (p_base * (1 - p_base) + p_obj * (1 - p_obj)) / (p_obj - p_base) ** 2
h = proportion_effectsize(p_obj, p_base)
n_sm = NormalIndPower().solve_power(effect_size=h, alpha=alfa, power=potencia, ratio=1)
n_grupo = int(np.ceil(n_sm))
dias = int(np.ceil(2 * n_grupo / visitantes_dia))
semanas = int(np.ceil(dias / 7))

assert all(v is not None for v in (za, zb, n_mano, h, n_sm, n_grupo, dias, semanas)), "Completa las líneas marcadas con TODO"
assert np.isclose(za, 1.959964, atol=1e-4) and np.isclose(zb, 1.281552, atol=1e-4), "za = norm.ppf(1 - alfa/2); zb = norm.ppf(potencia)"
ref_mano = (za + zb) ** 2 * (0.08 * 0.92 + 0.10 * 0.90) / 0.02 ** 2
assert np.isclose(n_mano, ref_mano), "revisa la fórmula: (za + zb)² · suma de varianzas / diferencia²"
assert np.isclose(h, proportion_effectsize(0.10, 0.08)), "proportion_effectsize(p_obj, p_base)"
assert abs(n_sm - n_mano) < 0.01 * n_mano, "n_sm y n_mano deberían diferir menos de un 1 %"
assert n_grupo == int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.10, 0.08), alpha=0.05, power=0.90, ratio=1))), "n_grupo = ceil(n_sm)"
assert dias == int(np.ceil(2 * n_grupo / 1000)) and semanas == int(np.ceil(dias / 7)), "días = ceil(2·n_grupo / visitantes_dia); semanas = ceil(días / 7)"
print(f"n a mano: {n_mano:.1f}   n statsmodels: {n_sm:.1f}   por grupo: {n_grupo}   días: {dias}   semanas: {semanas}")
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
n_a = n_b = 5000
conv_a, conv_b = 300, 370

pa, pb = conv_a / n_a, conv_b / n_b
dif = pb - pa
p_valor = proportions_ztest([conv_b, conv_a], [n_b, n_a])[1]
ee = np.sqrt(pa * (1 - pa) / n_a + pb * (1 - pb) / n_b)
ic_inf, ic_sup = dif - 1.96 * ee, dif + 1.96 * ee
lift = pb / pa - 1
significativo = bool(p_valor < 0.05)
incluye_bajo_mde = bool(ic_inf < 0.01)

assert all(v is not None for v in (dif, p_valor, ic_inf, ic_sup, lift, significativo, incluye_bajo_mde)), "Completa las líneas marcadas con TODO"
assert np.isclose(dif, 0.014), "dif = 370/5000 − 300/5000"
assert np.isclose(p_valor, proportions_ztest([370, 300], [5000, 5000])[1]), "proportions_ztest([conv_b, conv_a], [n_b, n_a]) y toma el segundo valor"
ee_ref = np.sqrt(0.06 * 0.94 / 5000 + 0.074 * 0.926 / 5000)
assert np.isclose(ic_inf, 0.014 - 1.96 * ee_ref) and np.isclose(ic_sup, 0.014 + 1.96 * ee_ref), "IC = dif ± 1,96 · error estándar sin combinar"
assert np.isclose(lift, 370 / 300 - 1), "lift = pB / pA − 1"
assert significativo == True and incluye_bajo_mde == True, "es significativo (p ≈ 0,005) pero el IC llega por debajo de 0,01"
print(f"Diferencia: {dif * 100:.2f} pp  p-value = {p_valor:.4f}  IC 95 % = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp  lift = {lift * 100:.1f} %")
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
rng_e3 = np.random.default_rng(21)
nsim, controles, por = 2000, 5, 768
xa = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
xb = rng_e3.binomial(por, 0.10, (nsim, controles)).cumsum(axis=1)
ns = por * np.arange(1, controles + 1)
pp = (xa + xb) / (2 * ns)
z = ((xb - xa) / ns) / np.sqrt(pp * (1 - pp) * 2 / ns)

sig = np.abs(z) > 1.96
tasa_peeking = sig.any(axis=1).mean()
tasa_final = sig[:, -1].mean()

assert all(v is not None for v in (sig, tasa_peeking, tasa_final)), "Completa las líneas marcadas con TODO"
assert sig.shape == (2000, 5) and sig.dtype == bool, "sig = np.abs(z) > 1.96 (matriz booleana 2000 × 5)"
assert np.isclose(tasa_peeking, (np.abs(z) > 1.96).any(axis=1).mean()), "usa sig.any(axis=1).mean(): significativo en algún control"
assert np.isclose(tasa_final, (np.abs(z[:, -1]) > 1.96).mean()), "usa sig[:, -1].mean(): solo el último control"
# Esperado: ≈ 5 % mirando una vez (± 3 errores estándar ≈ 1,5 pp) y ≈ 13 % con 5 controles
assert 0.035 < tasa_final < 0.065, "mirando solo al final la tasa debería rondar el 5 %"
assert 0.10 < tasa_peeking < 0.17, "con 5 controles la tasa real de falsos positivos debería rondar el 13 %"
print(f"Falsos positivos mirando una vez: {tasa_final * 100:.1f} %   mirando 5 veces: {tasa_peeking * 100:.1f} %")
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
obs_1 = np.array([25300, 24200])
obs_2 = np.array([24900, 24600])

chi2_1, p_1 = stats.chisquare(obs_1)
chi2_2, p_2 = stats.chisquare(obs_2)
srm_1 = bool(p_1 < 0.001)
srm_2 = bool(p_2 < 0.001)
esp_1 = obs_1.sum() / 2
chi2_1_mano = float(((obs_1 - esp_1) ** 2 / esp_1).sum())

assert all(v is not None for v in (chi2_1, p_1, chi2_2, p_2, srm_1, srm_2, chi2_1_mano)), "Completa las líneas marcadas con TODO"
assert np.isclose(chi2_1, 24.4444, atol=1e-3) and np.isclose(chi2_2, 1.8182, atol=1e-3), "stats.chisquare(observados) devuelve (chi2, p); el esperado por defecto es el reparto igual"
assert srm_1 == True and srm_2 == False, "el experimento 1 tiene SRM (p ≈ 0,0000008) y el 2 no (p ≈ 0,18)"
assert np.isclose(chi2_1_mano, chi2_1), "esperado = total / 2 = 24 750; suma de (obs − esp)² / esp para los dos grupos"
print(f"Experimento 1: chi2 = {chi2_1:.2f}  p = {p_1:.7f}  SRM = {srm_1}   |   Experimento 2: chi2 = {chi2_2:.2f}  p = {p_2:.3f}  SRM = {srm_2}")
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
tabla = pd.DataFrame({"segmento": ["nuevos", "nuevos", "recurrentes", "recurrentes"],
                      "grupo": ["A", "B", "A", "B"],
                      "usuarios": [200, 700, 800, 300],
                      "conversiones": [20, 84, 240, 96]})
conv = tabla.pivot(index="segmento", columns="grupo", values="conversiones")
usr = tabla.pivot(index="segmento", columns="grupo", values="usuarios")

tasa_seg = conv / usr
tasa_total = conv.sum() / usr.sum()
simpson = bool((tasa_seg["B"] > tasa_seg["A"]).all() and tasa_total["B"] < tasa_total["A"])
pesos = usr.sum(axis=1) / usr.values.sum()
estand = tasa_seg.mul(pesos, axis=0).sum()
dif_estand = float((estand["B"] - estand["A"]) * 100)

assert all(v is not None for v in (tasa_seg, tasa_total, simpson, dif_estand)), "Completa las líneas marcadas con TODO"
assert np.isclose(tasa_seg.loc["nuevos", "A"], 0.10) and np.isclose(tasa_seg.loc["recurrentes", "B"], 0.32), "tasa_seg = conv / usr (segmentos en filas, grupos en columnas)"
assert np.isclose(tasa_total["A"], 0.26) and np.isclose(tasa_total["B"], 0.18), "tasa total = conv.sum() / usr.sum()"
assert simpson == True, "B gana en los dos segmentos (12 % > 10 %, 32 % > 30 %) y pierde en el total (18 % < 26 %)"
assert np.isclose(dif_estand, 2.0), "pesos = usuarios del segmento (A + B) / total; tasa estandarizada = suma(peso · tasa del segmento)"
print(tasa_seg.round(3).to_string())
print("Total:", (tasa_total * 100).round(1).to_dict(), "  diferencia estandarizada (pp):", round(dif_estand, 2))
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
def generar_experimento(rng, n_a=5561, n_b=5548, p_a=0.08, p_b=0.095):
    """Datos sintéticos (ejemplo ilustrativo): una fila por visitante."""
    convierte = np.concatenate([rng.random(n_a) < p_a, rng.random(n_b) < p_b]).astype(int)
    return pd.DataFrame({"grupo": ["A"] * n_a + ["B"] * n_b, "convierte": convierte})

datos_reto = generar_experimento(np.random.default_rng(95))

n_plan = int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1)))
g = datos_reto.groupby("grupo")["convierte"].agg(["sum", "count"])
tamanos = {"A": int(g.loc["A", "count"]), "B": int(g.loc["B", "count"])}
p_srm = stats.chisquare([tamanos["A"], tamanos["B"]])[1]
p_valor = proportions_ztest([g.loc["B", "sum"], g.loc["A", "sum"]], [tamanos["B"], tamanos["A"]])[1]
ic_inf, ic_sup = confint_proportions_2indep(g.loc["B", "sum"], tamanos["B"], g.loc["A", "sum"], tamanos["A"], method="newcomb")
if p_srm < 0.001:
    decision = "SRM"
elif p_valor < 0.05 and ic_inf > 0:
    decision = "GO"
else:
    decision = "NO_GO"
informe = "Diseño: n = 5 561 por grupo para detectar +1,5 pp (8 % → 9,5 %) con α = 5 % y potencia 80 %. Controles: sin SRM. Resultado: B supera a A con IC que no incluye el 0. Decisión: lanzar B, sabiendo que el efecto plausible es modesto."

assert all(v is not None for v in (n_plan, tamanos, p_srm, p_valor, ic_inf, ic_sup, decision, informe)), "Completa las líneas marcadas con TODO"
ref_n = int(np.ceil(NormalIndPower().solve_power(effect_size=proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1)))
assert n_plan == ref_n, "n_plan = ceil(NormalIndPower().solve_power(proportion_effectsize(0.095, 0.08), alpha=0.05, power=0.80, ratio=1))"
g = datos_reto.groupby("grupo")["convierte"].agg(["sum", "count"])
assert tamanos == {"A": int(g.loc["A", "count"]), "B": int(g.loc["B", "count"])}, "tamanos = {'A': usuarios de A, 'B': usuarios de B}"
assert np.isclose(p_srm, stats.chisquare([g.loc["A", "count"], g.loc["B", "count"]])[1]), "stats.chisquare sobre los dos tamaños, reparto esperado igual"
ref_p = proportions_ztest([g.loc["B", "sum"], g.loc["A", "sum"]], [g.loc["B", "count"], g.loc["A", "count"]])[1]
assert np.isclose(p_valor, ref_p), "proportions_ztest([conv_B, conv_A], [n_B, n_A]) y toma el segundo valor"
ref_lo, ref_hi = confint_proportions_2indep(g.loc["B", "sum"], g.loc["B", "count"], g.loc["A", "sum"], g.loc["A", "count"], method="newcomb")
assert np.isclose(ic_inf, ref_lo) and np.isclose(ic_sup, ref_hi), "confint_proportions_2indep(conv_B, n_B, conv_A, n_A, method='newcomb')"
assert p_srm > 0.001 and p_valor < 0.05 and ic_inf > 0, "en estos datos: sin SRM, significativo, y el IC no incluye el 0"
assert decision == "GO", "aplica la regla: SRM → revisar; p < 0,05 e ic_inf > 0 → lanzar B; si no, no lanzar B"
assert isinstance(informe, str) and len(informe) >= 60, "escribe un párrafo de al menos 60 caracteres"
print(f"n_plan = {n_plan}   tamaños: {tamanos}   p_srm = {p_srm:.3f}   p-value = {p_valor:.4f}   IC = [{ic_inf * 100:.2f}, {ic_sup * 100:.2f}] pp")
print("Decisión:", decision)
print("Reto final correcto ✔")

---
## Autoevaluación

Anota en una celda de texto o en papel:

- **¿Qué ejercicios resolví sin ayuda?**
- **¿Qué concepto sigue poco claro?** (por ejemplo: fórmula de n, potencia, peeking, SRM, paradoja de Simpson)
- **Tiempo dedicado:** ___ minutos
- **Una pregunta que quiero llevar a la próxima clase:** ...